## Multilayer Perceptron For Image Dataset

In [19]:
import sys
print(sys.executable)
import torch
import torch.nn as nn 

/Library/Developer/CommandLineTools/usr/bin/python3


In [20]:
# select device CPU/GPU-CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


#### Standarization/Normalization

In [21]:
# data/image/computer vision preprocessing
from torchvision import transforms # transforms provides all data/image preprocessing funtionality for Computer Vision/Matrix data/Image data
transform = transforms.Compose([
    transforms.ToTensor(), # scales pixels values from 0-255 to 0-1 [scale down to avoid computational complexity] - normalization
    transforms.Normalize(mean=(0.1307,), std=(0.3081, )) # as the dataset is benchmark, so the mean and std is given by the dataset provider for 
                                                         # pixel value 0 to 1. So, first we scale down to 0-1, the use the mean, std normalization
                                                         # Othewise, we don't need to use the normalization. 
])

#### Import and Load Dataset [Practice dataset given by torch]

In [ ]:
# import torch practice datasets
from torchvision import datasets

train_dataset = datasets.MNIST(
    root='../../../../my-practice/Dataset/amnist_data',
    train=True,
    download=True,
    transform=transform # data preprocessing of a single batch done here.
)

test_dataset = datasets.MNIST(
    root='../../../../my-practice/Dataset/amnist_data',
    train=False,
    download=True,
    transform=transform
)

In [ ]:
# Load Dataset using DataLoader. DataLoader: Helps to load batch dataset. Create batches, Load data batch by batch to Dataset class for 
# preprocessed and then DataLoader provide the preprocessed data [ a single batch] to our model for calculation/prediction.
from torch.utils.data import DataLoader
train_loader = DataLoader(
    dataset=train_dataset, # provide single batch [data] to the dataset for preprocessing. Later, provide the preprocessed batch to our model
    batch_size=64, # create batches with 64 data [size]
    shuffle=True # When I want to distribute randomly
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=64,
    shuffle=False
)

#### Perceptron

In [24]:
class Perceptron(nn.Module):
    def __init__(self, input_size): # initialize the Perceptron using input_size and take nn.Module functionality.
        super(Perceptron, self).__init__()

        self.w = nn.Parameter(torch.randn(input_size)) # weights
        self.b = nn.Parameter(torch.randn(1)) # bias

    def forward(self, X):
        return X @ self.w + self.b # calculate prediction and return the results

In [25]:
input_size = 28 * 28 # according the dataset

#### Check sample result

In [26]:
# check sample result
perceptron = Perceptron(input_size)
sampe_input = torch.randn(input_size)
sample_output = perceptron(sampe_input)
print("Prediction = ", sample_output)

Prediction =  tensor([3.6963], grad_fn=<AddBackward0>)


#### Taking Activation Funtion [Introduced Non-linearity]

In [27]:
class RelU(nn.Module): # ReLU, result = max(0, input result of previous neuron/perceptron). Most used activation function. 
    def __init__(self):
        super(RelU, self).__init__() # initialization

    def forward(self, x):
        return torch.maximum(torch.tensor(0,0), x) # calculate result and reuturn

#### Hidden Linear Layer for multiple Perceptron

In [28]:
class Linear(nn.Module):
    def __init__(self, input_size, output_size): 
        super(Linear, self).__init__() # initialize using input and output size in this layer
        self.perceptrons = nn.ModuleList([ # define how many Perceptron will be in this layer. 
            Perceptron(input_size) for _ in range(output_size) # Input size of a singe Perceptron = total output comes from previous layer 
                                                               # total Perceptron in this layer == output_size.
        ])
    def forward(self, x): # x = input for this layer / a single output from previous layer
        outputs = [perceptron(x) for perceptron in self.perceptrons] # a single output(x) from previous layer went throuth all the perceptron in this layer. 
                                                                     # This is a input for all Perceptron in this layer
        outputs = torch.stack(outputs, dim=1) # creating matrix format from list format.  
        return outputs

#### Define Digit Classifier

In [29]:
class DigitClassifier(nn.Module):
    def __init__(self, input_size=784, output_size=10): # input size = 28*28, output = 10 digits [0 to 9]
        super(DigitClassifier, self).__init__() # Initialization
        self.fc1 = Linear(input_size, 512) # Linear(input, output) = Linear(784, 512). fc = Fully connected layer == Linear layer
        self.fc2 = Linear(512, 256) # Linear(input [previous layer output], output) = Linear (512, 256)
        self.fc3 = Linear(256, 128) # Linear(256, 128)
        self.fc4 = Linear(128, output_size) # Linear(128, 10 [final result]). Finally, we took total 4 Linear Layer to produce final prediction/output
        self.relu = nn.ReLU() # on ReLU [activation function] layer
    def forward(self, x): # x is an image for this dataset
        # print(x.shape) # B, 1, 28, 28. Where, B [Batch size], 1 [channel], 28 [H], 28 [W] - format of x [image]
        x = x.view(-1, input_size) # (B,784). Here, -1 used to conver the shape from (B, 1, 28, 28) to (B, 784)
        # How to conver: -1 does, B*1*28*28/784 = B. So, x = x.view(B, 784) and x.shape = (B, 784) - reshaped
        # print(x.shape)
        x = self.fc1(x) #  in first Linear = Linear(x) = Linear(B, 784)
        x = self.relu(x) # result went through activation function (ReLU)
        x = self.fc2(x) # result = Linear (512, 256)
        x = self.relu(x) # Relu(result)
        x = self.fc3(x) # result = (256, 128)
        x = self.relu(x) # ReLu(result)
        x = self.fc4(x) # final result/output/prediction = Linear(128, 10) - 10 digits from 0 to 9
        return x # return final result

#### Define Model

In [ ]:
""" Use of device

data is stored in (RAM/GPU) when program is running
model is also loaded (RAM/GPU)
"""
model = DigitClassifier(input_size=28 * 28, output_size=10).to(device) 
sampe_input = sampe_input.to(device) # model and sampe_input loaded in same device (GPU/RAM)

# check sample output result
output = model(sampe_input)
print(output.shape)
print(output)

torch.Size([1, 10])
tensor([[-28588.4961, -53161.4648, -71203.0469, -32321.2930, -27269.7676,
          86896.2734,   9270.3574,  20582.0605, -12890.0723,  32103.7461]],
       grad_fn=<StackBackward0>)


##### Visualized model's parameters

In [ ]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

Total trainable parameters: 567,434


In [ ]:
(784+1) * 512 + (512+1) * 256 + (256+1) * 128+ (128+1) * 10 # multiplication of all input in each layer(w) + bias(b) = Total trainable parameters

567434

In [ ]:
param_size = next(model.parameters()).element_size() # total bytes
model_size_bytes = total_params * param_size
model_size_mb = model_size_bytes / (1024 * 1024)

In [ ]:
print("\n" + "="*50)
print(f"Total trainable parameters : {total_params:,}")
print(f"Parameter size (float32)   : {param_size} bytes")
print(f"Estimated model size       : {model_size_mb:.2f} MB")
print(f"Estimated size (float16)   : {model_size_mb/2:.2f} MB")


Total trainable parameters : 567,434
Parameter size (float32)   : 4 bytes
Estimated model size       : 2.16 MB
Estimated size (float16)   : 1.08 MB


In [36]:
max_val, predicted_id = torch.max(output, 1)
print(max_val)
print("Class label:", predicted_id)

tensor([86896.2734], grad_fn=<MaxBackward0>)
Class label: tensor([5])


#### Define Optimizer and Loss/Cost Calculator

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001) # used for optimize the loss/cost
criterion = nn.CrossEntropyLoss() # Used for non-binary classification

#### Training and Evaluation

In [ ]:
num_epochs = 2

for epoch in range(num_epochs):
    model.train() # change model mode
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader): # loading a batch one by one. Here, batch_idx -> index, (data, target) -> data of the batch
        data, target = data.to(device), target.to(device) # ensures model and data are on the same device. i.e. device is of two type cpu, cuda
        outputs = model(data) # prediction
        loss = criterion(outputs, target) # calculate loss
        optimizer.zero_grad() # finding local minimum. [optimal/minimum loss points]
        loss.backward() # Compute Gradient Descent(Back Propagation)
        optimizer.step() # update weights(w) and bias(b) using optimal points
        running_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}')
    
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

Epoch [1/2], Step [1/938], Loss: 86821.7188
Epoch [1/2], Step [101/938], Loss: 6516.8730
Epoch [1/2], Step [201/938], Loss: 4018.4497
Epoch [1/2], Step [301/938], Loss: 3098.9648
Epoch [1/2], Step [401/938], Loss: 2241.2437
Epoch [1/2], Step [501/938], Loss: 1168.8379
Epoch [1/2], Step [601/938], Loss: 716.7803
Epoch [1/2], Step [701/938], Loss: 1130.4253
Epoch [1/2], Step [801/938], Loss: 1444.4124
Epoch [1/2], Step [901/938], Loss: 449.0769
Epoch 1, Loss: 3273.2934522811793
Epoch [2/2], Step [1/938], Loss: 713.2571
Epoch [2/2], Step [101/938], Loss: 546.4108
Epoch [2/2], Step [201/938], Loss: 408.4341
Epoch [2/2], Step [301/938], Loss: 185.7540
Epoch [2/2], Step [401/938], Loss: 868.0889
Epoch [2/2], Step [501/938], Loss: 467.2033
Epoch [2/2], Step [601/938], Loss: 470.9731
Epoch [2/2], Step [701/938], Loss: 975.4496
Epoch [2/2], Step [801/938], Loss: 311.8623
Epoch [2/2], Step [901/938], Loss: 262.6411
Epoch 2, Loss: 678.8313879895566


In [ ]:
model.eval() # change model mode from training [default] to evaluation
correct = 0
total = 0
with torch.no_grad(): # no need gradient calculation in evaluation part
    for data, target in test_loader: # loading test data
        data, target = data.to(device), target.to(device) # taking same device [cpu, cuda/gpu]
        outputs = model(data) # prediction
        _, predicted = torch.max(outputs.data, 1) # max predicted label
        total += target.size(0)
        correct += (predicted == target).sum().item() # see predicted label == target. Then, sum up as a correct prediction.

accuracy = 100 * correct / total # total accuracy
print(f'Accuracy on the test set: {accuracy:.2f}%')

Accuracy on the test set: 90.90%


## Tutorial: Multilayer Perceptron Explained (বাংলা)

এই tutorial অংশটি notebook-এর শেষের পরে যোগ করা হলো, যাতে একজন beginner learner পুরো notebook-এর code, concept, formula, math intuition, এবং purpose step by step বুঝতে পারে।

---

## 1. এই notebook-এর main goal কী?

এই notebook-এর মূল উদ্দেশ্য হলো **Multilayer Perceptron (MLP)** ব্যবহার করে image classification শেখানো। এখানে dataset হিসেবে `MNIST` digit dataset ব্যবহার করা হয়েছে, যেখানে model-এর কাজ হলো 0 থেকে 9 পর্যন্ত handwritten digit classify করা।

সহজভাবে বললে, এই notebook দেখাচ্ছে কীভাবে:

- image data preprocess করতে হয়
- dataset load করতে হয়
- batch-wise data feed করতে হয়
- single perceptron থেকে multi-layer network বানাতে হয়
- hidden layer এবং activation function ব্যবহার করতে হয়
- model-এর total trainable parameter count করতে হয়
- loss function ও optimizer define করতে হয়
- training loop লিখতে হয়
- test set-এ model evaluate করতে হয়

---

## 2. Multilayer Perceptron (MLP) কী?

Perceptron হলো neural network-এর basic unit। কিন্তু একটি single perceptron complex pattern শিখতে পারে না। তাই একাধিক perceptron layer-wise arrange করলে তাকে **Multilayer Perceptron** বলা হয়।

### Basic structure

`Input Layer -> Hidden Layer 1 -> Hidden Layer 2 -> ... -> Output Layer`

এই notebook-এ model architecture roughly এমন:

`784 -> 512 -> 256 -> 128 -> 10`

এখানে:

- `784` = input image flatten করার পর total pixel feature
- `512, 256, 128` = hidden layer neuron count
- `10` = output class count (digit 0-9)

### Formula intuition

প্রতিটি layer-এর কাজ:

\[z = XW + b\]
\[a = f(z)\]

এখানে:

- `X` = input
- `W` = weight matrix
- `b` = bias
- `f` = activation function (এখানে ReLU)

---

## 3. Notebook Cell by Cell Explanation

### Cell 0
`## Multilayer Perceptron For Image Dataset`

এটি title cell। এটি বলে দিচ্ছে যে notebook-এর focus হলো image dataset-এর জন্য multilayer perceptron।

### Cell 1
```python
import sys
print(sys.executable)
import torch
import torch.nn as nn
```

এখানে environment path print করা হচ্ছে এবং PyTorch import করা হচ্ছে।

- `sys.executable` দিয়ে বোঝা যায় কোন Python environment চলছে
- `torch` হলো core deep learning library
- `torch.nn` neural network layer, module, loss function define করার জন্য

### Cell 2
```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

এখানে model training CPU নাকি GPU-তে হবে সেটা select করা হচ্ছে।

#### Why important?

- `cpu` ধীরে training করে
- `cuda` থাকলে GPU use করে অনেক দ্রুত training করা যায়

এই notebook ভালো practice দেখিয়েছে: model, input data, target সব same device-এ নিতে হবে।

### Cell 3-4: Standardization / Normalization

```python
from torchvision import transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081, ))
])
```

এখানে image preprocessing define করা হয়েছে।

#### `transforms.ToTensor()`

এটি image pixel value-কে 0-255 range থেকে 0-1 range-এ নিয়ে আসে।

গাণিতিকভাবে:
\[x_{scaled} = \frac{x}{255}\]

উদাহরণ:
- pixel = 255 হলে scaled value = 1.0
- pixel = 128 হলে scaled value প্রায় 0.502

#### `transforms.Normalize(mean, std)`

এটি standardized input তৈরি করে। Formula:
\[x_{norm} = \frac{x - mean}{std}\]

MNIST dataset-এর benchmark mean এবং std দেওয়া আছে, তাই সেগুলো ব্যবহার করা হচ্ছে।

#### Why normalization useful?

- optimization stable হয়
- gradient better behave করে
- training দ্রুত converge করতে পারে

### Cell 5-6: Dataset Load

```python
train_dataset = datasets.MNIST(..., train=True, ...)
test_dataset = datasets.MNIST(..., train=False, ...)
```

এখানে `MNIST` dataset load করা হয়েছে।

MNIST dataset-এ থাকে:

- 28x28 grayscale handwritten digit image
- label 0 থেকে 9 পর্যন্ত

#### Train বনাম Test

- `train=True` → training data
- `train=False` → evaluation/test data

এটি খুব important, কারণ training data দিয়ে model শেখে আর test data দিয়ে আমরা generalization measure করি।

### Cell 7: DataLoader

```python
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)
```

`DataLoader` batch-wise data load করতে সাহায্য করে।

#### `batch_size=64`
মানে একবারে 64টি image model-এ যাবে।

#### `shuffle=True`
Training-এর সময় data random order-এ দিতে useful, যাতে model sequence memorize না করে।

#### `shuffle=False`
Test-এর সময় সাধারণত shuffle না করলেও সমস্যা নেই, কারণ এখানে শুধু evaluation হচ্ছে।


## 4. Perceptron, ReLU, Linear Layer, এবং Classifier Architecture

### Cell 8-9: Perceptron Class

```python
class Perceptron(nn.Module):
    def __init__(self, input_size):
        super(Perceptron, self).__init__()
        self.w = nn.Parameter(torch.randn(input_size))
        self.b = nn.Parameter(torch.randn(1))

    def forward(self, X):
        return X @ self.w + self.b
```

এখানে একটি custom perceptron define করা হয়েছে।

#### Weight এবং Bias

- `self.w` = trainable weight
- `self.b` = trainable bias

Perceptron formula:
\[z = XW + b\]

এই class এখনো activation apply করছে না। এটি শুধু linear transformation করছে।

#### Beginner intuition

ধরি input vector `x = [x1, x2, x3]`। তাহলে perceptron output হবে:
\[z = x_1w_1 + x_2w_2 + x_3w_3 + b\]

এটি basically weighted sum।

### Cell 10
`input_size = 28 * 28`

MNIST image-এর shape 28x28। Flatten করলে total feature হয়:
\[28 \times 28 = 784\]

তাই input_size = 784।

### Cell 11-12: Sample Result Check

এখানে random input দিয়ে perceptron test করা হয়েছে।

Purpose:

- class কাজ করছে কি না
- input-output flow ঠিক আছে কি না

### Cell 13-14: Activation Function (ReLU)

```python
class RelU(nn.Module):
    def forward(self, x):
        return torch.maximum(torch.tensor(0,0), x)
```

এখানে custom ReLU define করার চেষ্টা করা হয়েছে। Conceptually ReLU formula হলো:
\[ReLU(x) = max(0, x)\]

#### Example

- `ReLU(-5) = 0`
- `ReLU(3.2) = 3.2`

#### Why ReLU?

- non-linearity introduce করে
- negative value suppress করে
- deep learning-এ খুব popular

Note: পরে model-এ built-in `nn.ReLU()` ব্যবহার করা হয়েছে, যা better practice।

### Cell 15-16: Hidden Linear Layer

```python
class Linear(nn.Module):
    def __init__(self, input_size, output_size):
        self.perceptrons = nn.ModuleList([
            Perceptron(input_size) for _ in range(output_size)
        ])
```

এখানে একটি full linear layer manually বানানো হয়েছে, যেখানে `output_size` সংখ্যক perceptron রাখা হয়েছে।

ধরি:
- `input_size = 784`
- `output_size = 512`

তাহলে এই layer-এ 512টি perceptron থাকবে, এবং প্রতিটি perceptron 784-dimensional input নেবে।

#### `nn.ModuleList` কেন?

কারণ PyTorch যেন এই perceptron-গুলোকে trainable submodule হিসেবে track করতে পারে।

#### Forward logic

```python
outputs = [perceptron(x) for perceptron in self.perceptrons]
outputs = torch.stack(outputs, dim=1)
```

একই input সব perceptron-এর মধ্যে যাচ্ছে, তারপর তাদের output stack করে matrix বানানো হচ্ছে।

এটি manual fully connected layer construction-এর খুব ভালো educational example।

### Cell 17-18: DigitClassifier

এটি পুরো MLP architecture define করছে।

```python
self.fc1 = Linear(784, 512)
self.fc2 = Linear(512, 256)
self.fc3 = Linear(256, 128)
self.fc4 = Linear(128, 10)
```

এখানে 4টি fully connected layer আছে।

Flow:
`784 -> 512 -> 256 -> 128 -> 10`

#### `x.view(-1, input_size)`

Original image shape হয় `(B, 1, 28, 28)`। এখানে:
- `B` = batch size
- `1` = grayscale channel
- `28x28` = image size

MLP image matrix directly নেয় না, flattened vector নেয়। তাই reshape করা হয়:
\[(B,1,28,28) \rightarrow (B,784)\]

#### ReLU placement

প্রতিটি hidden linear layer-এর পরে ReLU দেওয়া হয়েছে:

- `fc1 -> relu`
- `fc2 -> relu`
- `fc3 -> relu`
- `fc4` final logits

#### Why output layer-এ ReLU নেই?

কারণ final layer raw score বা **logit** দিচ্ছে, যা পরে `CrossEntropyLoss` internally handle করবে।

### Cell 19-20: Define Model

```python
model = DigitClassifier(...).to(device)
```

এখানে model create করে selected device-এ পাঠানো হয়েছে।

তারপর sample input-ও একই device-এ নেওয়া হয়েছে।

#### Important rule

PyTorch-এ model CPU-তে আর input GPU-তে থাকলে error হবে। তাই model, data, target সব same device-এ থাকতে হবে।


## 5. Parameters, Loss, Optimizer, Training, Evaluation

### Cell 21-25: Model Parameter Visualization

```python
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
```

এখানে total trainable parameter count করা হয়েছে।

#### `p.numel()`
একটি tensor-এ মোট কতটি element আছে তা বলে।

#### Manual formula
পরের cell-এ manual হিসাব করা হয়েছে:
\[(784+1) \times 512 + (512+1) \times 256 + (256+1) \times 128 + (128+1) \times 10\]

এখানে প্রতিটি layer-এর জন্য:
- `(input + 1)` → bias সহ total trainable unit per neuron
- তারপর `× output_neuron_count`

#### Example
প্রথম layer:
\[(784 + 1) \times 512\]
কারণ 784 weight + 1 bias per neuron, মোট 512 neuron।

#### Model size estimate
পরের cell-এ parameter size bytes-এ convert করে model size MB-তে estimate করা হয়েছে।

যদি float32 হয়, সাধারণত 1 parameter = 4 bytes।

### Cell 26: Predicted Class

```python
max_val, predicted_id = torch.max(output, 1)
```

এখানে final output logits থেকে highest score-এর class নেওয়া হচ্ছে।

#### Why max?
কারণ classification-এ সাধারণত যে class-এর score সবচেয়ে বেশি, সেটিকে predicted class ধরা হয়।

### Cell 27-28: Optimizer and Loss

```python
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
```

#### `CrossEntropyLoss` কেন?

এটি multi-class classification-এর জন্য standard loss function। MNIST-এ 10টি class আছে, তাই এটি সঠিক choice।

Cross entropy conceptually softmax + negative log likelihood combine করে।

Softmax formula:
\[P(y=i) = \frac{e^{z_i}}{\sum_j e^{z_j}}\]

তারপর true class probability-এর negative log নেওয়া হয়।

#### Adam optimizer
Adam adaptive optimizer, practical deep learning-এ খুব common। এটি দ্রুত এবং stable training-এ সাহায্য করে।

### Cell 29-30: Training Loop

এখানে পুরো training process লেখা হয়েছে।

#### Step 1: `model.train()`
Model-কে train mode-এ নেয়। Dropout/BatchNorm থাকলে এই mode important।

#### Step 2: batch loop
`for batch_idx, (data, target) in enumerate(train_loader)`

এখানে DataLoader batch-by-batch image ও label দিচ্ছে।

#### Step 3: device transfer
`data, target = data.to(device), target.to(device)`

Model আর data same device-এ ensure করা হচ্ছে।

#### Step 4: prediction
`outputs = model(data)`

এখানে batch input দিয়ে forward pass হচ্ছে। Output shape সাধারণত `(batch_size, 10)` হবে।

#### Step 5: loss
`loss = criterion(outputs, target)`

Prediction logits আর actual class label compare করে loss বের করা হচ্ছে।

#### Step 6: old gradient clear
`optimizer.zero_grad()`

PyTorch gradient accumulate করে, তাই নতুন step-এর আগে clear করতে হয়।

#### Step 7: backpropagation
`loss.backward()`

এখানে autograd সব trainable parameter-এর gradient বের করে।

#### Step 8: update
`optimizer.step()`

Optimizer gradient ব্যবহার করে weight ও bias update করে।

#### Step 9: running loss
Epoch-এর মধ্যে average loss track করা হচ্ছে, যাতে model improvement monitor করা যায়।

### Cell 31: Evaluation on Test Set

```python
model.eval()
with torch.no_grad():
    ...
```

এখানে model test set-এ evaluate করা হচ্ছে।

#### `model.eval()`
evaluation mode-এ নেয়।

#### `torch.no_grad()`
gradient calculation বন্ধ রাখে, তাই evaluation faster হয় এবং memory কম লাগে।

#### Prediction logic
`_, predicted = torch.max(outputs.data, 1)`

এখানে 10 class score-এর মধ্যে highest score-এর class নেওয়া হচ্ছে।

#### Accuracy formula
\[Accuracy = \frac{Correct\ Predictions}{Total\ Samples} \times 100\]\n

যদি 10000 test image-এর মধ্যে 9450টি সঠিক predict হয়, তাহলে accuracy হবে:
\[\frac{9450}{10000} \times 100 = 94.5\%\]

---

## 6. Beginner-Friendly Big Picture

এই notebook-এর পুরো flow এক লাইনে:

`Image -> Tensor -> Normalize -> Flatten -> Linear Layers -> ReLU -> Final 10 Scores -> CrossEntropyLoss -> Backprop -> Accuracy`

আরও simple flow:

```text
MNIST Image
-> Preprocessing
-> Batch Loading
-> MLP Model
-> Forward Pass
-> Loss Calculation
-> Gradient Calculation
-> Weight Update
-> Test Prediction
-> Accuracy
```

---

## 7. এই notebook থেকে কী কী concept শেখা হলো

- device selection (CPU/GPU)
- image normalization
- torchvision dataset usage
- DataLoader batching
- perceptron concept
- hidden layer construction
- multilayer perceptron architecture
- ReLU activation
- parameter counting
- model memory estimation
- multi-class classification
- CrossEntropyLoss
- Adam optimizer
- training loop
- evaluation loop
- test accuracy

---

## 8. Real-world Use Case

MLP image classification-এর foundation হিসেবে useful, especially beginner level-এ। যদিও modern computer vision-এ CNN বেশি effective, তবুও MLP দিয়ে beginner learner সহজে neural network-এর fundamental বুঝতে পারে।

MNIST-এর মতো dataset-এ MLP শেখা future topic-এর foundation: 

- CNN
- deep feedforward network
- regularization
- dropout
- batch normalization

---

## 9. Improvement Ideas

এই notebook future-এ improve করতে চাইলে নিচের জিনিস যোগ করা যেতে পারে:

- validation set add করা
- loss curve plot করা
- confusion matrix add করা
- per-class accuracy দেখা
- dropout layer add করা
- built-in `nn.Linear` use করে compare করা
- model save/load add করা
- prediction visualization add করা

---

## 10. Final Takeaway

এই notebook beginner-কে খুব important একটি ধারণা দেয়: **একটি image classification model শুধু dataset load করলেই হয় না, বরং preprocessing, batching, architecture design, loss, optimizer, training loop, evaluation - সবকিছু মিলে পুরো deep learning pipeline তৈরি হয়।**

যদি তুমি এই notebook-এর flow বুঝে ফেলো, তাহলে next step হিসেবে CNN, dropout, batch normalization, learning curve, এবং advanced evaluation শিখতে অনেক সহজ হবে।
